<a href="https://colab.research.google.com/github/mlbyhoang/marketing-attribution/blob/main/notebooks/12_shapley_attribution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 12 – Shapley Attribution

This notebook applies Shapley value-based attribution to interpret the predictions of the trained Dual-Attention GRU (DAGRU) model from Notebook 10.

The analysis focuses on:

- Estimating touchpoint contributions using Shapley values
- Aggregating attribution credits at the marketing channel level
- Comparing Shapley attribution with rule-based attribution models
- Exploring an illustrative budget allocation scenario

The analysis reuses the trained DAGRU model and its prepared test data without retraining.

**Objective:** To investigate how individual marketing touchpoints contribute to the model's predicted conversion probabilities.


In [ ]:
# Load libraries
import os
import json
import math
import time
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from google.colab import drive, userdata


In [ ]:
# Mount Google Drive
drive.mount("/content/drive")

# Configure artifact directories
results_path = userdata.get("MTA_RESULTS_PATH")
model_path = userdata.get("MTA_MODEL_PATH")

if not results_path or not model_path:
    raise ValueError(
        "Missing MTA_RESULTS_PATH or MTA_MODEL_PATH Colab secret."
    )

# Define DAGRU artifact paths
test_inputs_path = os.path.join(
    results_path, "dagru_test_inputs.npz"
)

channel_classes_path = os.path.join(
    results_path, "dagru_channel_classes.json"
)

dagru_model_path = os.path.join(
    model_path, "ablation_3_final.keras"
)

print("DAGRU artifact paths configured.")


In [ ]:
# Load prepared DAGRU test inputs
with np.load(test_inputs_path) as data:
    X_seq_test = data["X_seq_test"]
    X_time_test = data["X_time_test"]
    X_num_test = data["X_num_test"]
    y_test = data["y_test"]

# Load original channel vocabulary
with open(channel_classes_path, "r", encoding="utf-8") as f:
    channel_classes = json.load(f)

# Reconstruct channel mappings
channel_to_id = {
    channel: idx + 1
    for idx, channel in enumerate(channel_classes)
}

id_to_channel = {
    idx: channel
    for channel, idx in channel_to_id.items()
}

# Validate input dimensions
n_journeys = len(X_seq_test)

assert all(
    len(arr) == n_journeys
    for arr in [X_time_test, X_num_test, y_test]
), "Inconsistent number of test journeys."

assert X_seq_test.shape[1] == X_time_test.shape[1], (
    "Sequence and time inputs have different lengths."
)

assert np.all(
    (X_seq_test >= 0) &
    (X_seq_test <= len(channel_classes))
), "Invalid channel IDs detected."

print("===== DAGRU Test Data =====")
print(f"Test journeys: {n_journeys:,}")
print(f"Channel sequences: {X_seq_test.shape}")
print(f"Time features: {X_time_test.shape}")
print(f"Numerical features: {X_num_test.shape}")
print(f"Targets: {y_test.shape}")
print(f"Marketing channels: {len(channel_classes)}")


In [ ]:
# Register custom layers required by the trained DAGRU model
custom_objects = {
    "AttentionSoftmax": AttentionSoftmax,
    "WeightedSum": WeightedSum,
    "DualAttentionFusion": DualAttentionFusion,
    "ZeroStateLayer": ZeroStateLayer,
    "SliceLayer": SliceLayer,
    "DecoderGRUCell": DecoderGRUCell
}

# Load trained DAGRU model without compiling
model = tf.keras.models.load_model(
    dagru_model_path,
    custom_objects=custom_objects,
    compile=False
)

print("===== Trained DAGRU Model =====")
print(f"Model: {model.name}")
print(f"Inputs: {len(model.inputs)}")
print(f"Outputs: {len(model.outputs)}")

# Validate model input dimensions
expected_shapes = [
    X_seq_test.shape[1:],
    X_time_test.shape[1:],
    X_num_test.shape[1:]
]

for model_input, expected_shape in zip(
    model.inputs, expected_shapes
):
    actual_shape = tuple(model_input.shape[1:])

    if actual_shape != expected_shape:
        raise ValueError(
            f"Input shape mismatch: "
            f"expected {actual_shape}, got {expected_shape}"
        )

print("Model input dimensions validated.")


In [ ]:
# Define attribution sample criteria
max_touchpoints = 10

# Count non-padding touchpoints per journey
journey_lengths = np.count_nonzero(X_seq_test, axis=1)

# Select converting journeys with 1 to 10 touchpoints
attribution_mask = (
    (np.asarray(y_test).reshape(-1) == 1)
    & (journey_lengths >= 1)
    & (journey_lengths <= max_touchpoints)
)

# Extract attribution sample
X_seq_attr = X_seq_test[attribution_mask]
X_time_attr = X_time_test[attribution_mask]
X_num_attr = X_num_test[attribution_mask]

print("===== Attribution Sample =====")
print(f"Total test journeys: {len(X_seq_test):,}")
print(f"Converting test journeys: {np.sum(np.asarray(y_test).reshape(-1) == 1):,}")
print(f"Selected journeys: {len(X_seq_attr):,}")
print(f"Maximum touchpoints: {max_touchpoints}")


In [ ]:
def compute_shapley_values(model, seq, time_features, num_features):
    """
    Compute exact Shapley values for a single customer journey
    using channel-token deletion.
    """
    positions = np.flatnonzero(seq != 0)
    n = len(positions)

    if n == 0:
        return np.zeros_like(seq, dtype=float)

    # Generate all possible coalitions
    coalitions = []
    for size in range(n + 1):
        coalitions.extend(combinations(range(n), size))

    # Create model inputs for each coalition
    seq_batch = np.zeros(
        (len(coalitions), len(seq)),
        dtype=seq.dtype
    )

    for i, coalition in enumerate(coalitions):
        selected_positions = positions[list(coalition)]
        seq_batch[i, selected_positions] = seq[selected_positions]

    time_batch = np.repeat(
        time_features[np.newaxis, ...],
        len(coalitions),
        axis=0
    )

    num_batch = np.repeat(
        num_features[np.newaxis, ...],
        len(coalitions),
        axis=0
    )

    # Predict conversion probabilities
    predictions = model.predict(
        [seq_batch, time_batch, num_batch],
        batch_size=256,
        verbose=0
    )

    # DAGRU conversion probability is the second output
    conversion_probs = np.asarray(predictions[1]).reshape(-1)

    coalition_values = {
        frozenset(coalition): prob
        for coalition, prob in zip(coalitions, conversion_probs)
    }

    # Compute exact Shapley contributions
    shapley = np.zeros(len(seq), dtype=float)
    factorial_n = math.factorial(n)

    for j, position in enumerate(positions):
        other_players = [k for k in range(n) if k != j]

        for size in range(n):
            weight = (
                math.factorial(size)
                * math.factorial(n - size - 1)
                / factorial_n
            )

            for subset in combinations(other_players, size):
                subset = frozenset(subset)

                marginal_contribution = (
                    coalition_values[subset | {j}]
                    - coalition_values[subset]
                )

                shapley[position] += weight * marginal_contribution

    return shapley


In [ ]:
# Calculate raw Shapley values for selected journeys
start_time = time.time()

shapley_values = []

for i, (seq, time_features, num_features) in enumerate(
    zip(X_seq_attr, X_time_attr, X_num_attr, strict=True)
):
    values = compute_shapley_values(
        model=model,
        seq=seq,
        time_features=time_features,
        num_features=num_features
    )

    shapley_values.append(values)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1:,} / {len(X_seq_attr):,} journeys")

shapley_values = np.asarray(shapley_values)

elapsed_time = time.time() - start_time

print("===== Shapley Attribution =====")
print(f"Processed journeys: {len(shapley_values):,}")
print(f"Shapley values shape: {shapley_values.shape}")
print(f"Execution time: {elapsed_time / 60:.2f} minutes")


In [ ]:
# Save raw Shapley attribution results
shapley_path = os.path.join(
    results_path,
    "dagru_shapley_raw.npz"
)

np.savez_compressed(
    shapley_path,
    shapley_values=shapley_values,
    X_seq_attr=X_seq_attr
)

print("Raw Shapley attribution results saved.")


In [ ]:
def normalize_shapley_values(shapley_values):
    """
    Normalize positive Shapley contributions per journey.

    Journeys without positive contributions receive zero credits.
    """
    positive_values = np.maximum(shapley_values, 0)

    totals = positive_values.sum(axis=1, keepdims=True)

    normalized_values = np.divide(
        positive_values,
        totals,
        out=np.zeros_like(positive_values, dtype=float),
        where=totals > 0
    )

    return normalized_values


# Calculate normalized attribution credits
shapley_normalized = normalize_shapley_values(
    shapley_values
)

# Validate attribution credits
credit_sums = shapley_normalized.sum(axis=1)

print("===== Normalized Attribution =====")
print(f"Journeys: {len(shapley_normalized):,}")
print(f"Journeys with positive credits: {np.sum(credit_sums > 0):,}")
print(f"Journeys without positive credits: {np.sum(credit_sums == 0):,}")

assert np.allclose(
    credit_sums,
    np.where(credit_sums > 0, 1.0, 0.0)
), "Invalid attribution credit sums."


In [ ]:
# Aggregate normalized Shapley credits by marketing channel
channel_credits = {}

for seq, credits in zip(
    X_seq_attr,
    shapley_normalized,
    strict=True
):
    for channel_id, credit in zip(seq, credits, strict=True):

        # Ignore padding tokens
        if channel_id == 0:
            continue

        channel = id_to_channel[int(channel_id)]

        channel_credits[channel] = (
            channel_credits.get(channel, 0.0) + float(credit)
        )

# Create channel-level attribution table
df_channel_attribution = pd.DataFrame(
    channel_credits.items(),
    columns=["channel", "total_credit"]
)

# Calculate relative attribution shares
total_credit = df_channel_attribution["total_credit"].sum()

df_channel_attribution["attribution_share"] = (
    df_channel_attribution["total_credit"] / total_credit
    if total_credit > 0
    else 0.0
)

# Sort channels by attribution share
df_channel_attribution = (
    df_channel_attribution
    .sort_values("attribution_share", ascending=False)
    .reset_index(drop=True)
)

print("===== Channel Attribution =====")
print(
    df_channel_attribution.to_string(
        index=False,
        formatters={
            "total_credit": "{:.2f}".format,
            "attribution_share": "{:.2%}".format
        }
    )
)


In [ ]:
# Prepare channel attribution data
df_plot = df_channel_attribution.sort_values(
    "attribution_share",
    ascending=True
)

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    df_plot["channel"],
    df_plot["attribution_share"] * 100
)

# Add percentage labels
ax.bar_label(
    bars,
    fmt="%.1f%%",
    padding=4
)

# Format chart
ax.set_title(
    "DAGRU – Shapley Attribution by Marketing Channel",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Attribution Share (%)")
ax.set_ylabel("Marketing Channel")

ax.set_xlim(
    0,
    max(df_plot["attribution_share"].max() * 110, 0.01) * 100
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:

def calculate_rule_based_attribution(sequences):
    """
    Calculate First-Touch, Last-Touch, and Linear attribution
    for encoded customer journeys.
    """
    results = {
        "First-Touch": {},
        "Last-Touch": {},
        "Linear": {}
    }

    for seq in sequences:
        # Exclude padding tokens
        touchpoints = seq[seq != 0]
        n = len(touchpoints)

        if n == 0:
            continue

        # First-Touch attribution
        first_channel = id_to_channel[int(touchpoints[0])]
        results["First-Touch"][first_channel] = (
            results["First-Touch"].get(first_channel, 0) + 1
        )

        # Last-Touch attribution
        last_channel = id_to_channel[int(touchpoints[-1])]
        results["Last-Touch"][last_channel] = (
            results["Last-Touch"].get(last_channel, 0) + 1
        )

        # Linear attribution
        for channel_id in touchpoints:
            channel = id_to_channel[int(channel_id)]

            results["Linear"][channel] = (
                results["Linear"].get(channel, 0) + 1 / n
            )

    return results


# Calculate rule-based attribution
rule_based_results = calculate_rule_based_attribution(
    X_seq_attr
)

# Combine attribution methods
attribution_comparison = {
    "Shapley": dict(
        zip(
            df_channel_attribution["channel"],
            df_channel_attribution["attribution_share"]
        )
    )
}

for method, channel_credits in rule_based_results.items():
    total = sum(channel_credits.values())

    attribution_comparison[method] = {
        channel: credit / total
        for channel, credit in channel_credits.items()
    } if total > 0 else {}

# Create comparison table
df_comparison = pd.DataFrame(
    attribution_comparison
).fillna(0.0)

df_comparison = (
    df_comparison
    .rename_axis("channel")
    .reset_index()
    .sort_values("Shapley", ascending=False)
    .reset_index(drop=True)
)

print("===== Attribution Model Comparison =====")
print(
    df_comparison.to_string(
        index=False,
        formatters={
            method: "{:.2%}".format
            for method in attribution_comparison
        }
    )
)


In [ ]:
# Prepare attribution comparison data
methods = ["Shapley", "First-Touch", "Last-Touch", "Linear"]

df_plot = df_comparison.set_index("channel")[methods] * 100

# Sort channels by Shapley attribution share
df_plot = df_plot.sort_values(
    "Shapley",
    ascending=True
)

# Create grouped horizontal bar chart
ax = df_plot.plot(
    kind="barh",
    figsize=(12, 7),
    width=0.8
)

# Format chart
ax.set_title(
    "DAGRU – Comparison of Attribution Models",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Attribution Share (%)")
ax.set_ylabel("Marketing Channel")

ax.legend(
    title="Attribution Method",
    loc="lower right"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


In [ ]:
# Define hypothetical marketing budget
total_budget = 100_000

methods = [
    "Shapley",
    "First-Touch",
    "Last-Touch",
    "Linear"
]

# Calculate budget allocation by attribution method
df_budget = df_comparison[["channel"]].copy()

for method in methods:
    df_budget[method] = (
        df_comparison[method] * total_budget
    )

# Sort by Shapley-based allocation
df_budget = (
    df_budget
    .sort_values("Shapley", ascending=False)
    .reset_index(drop=True)
)

print("===== Illustrative Budget Allocation =====")
print(f"Total budget: €{total_budget:,.2f}")

print(
    df_budget.to_string(
        index=False,
        formatters={
            method: "€{:,.2f}".format
            for method in methods
        }
    )
)
